> **Parallel set, notebook 8 of 10.** Shard 0 of 3 of NB3; P0 only (add `'P1'` to `TIERS` once P0 is finished). **Canary: runs the pilot.** Read `aml_pilot/PILOT_REPORT.md`; the other notebooks of this kind should start once it says PASS.
>
> P0 jobs in this notebook: `nb3-tdgnn-HI-Small-H0-shuffled-s0`, `nb3-tdgnn-HI-Small-H0-s1`, `nb3-tdgnn-HI-Small-H1-s1`, `nb3-tdgnn-HI-Small-H3600-s1`, `nb3-tdgnn-HI-Small-H86400-s1`, `nb3-tdgnn-HI-Small-Hinf-s1`, `nb3-tdgnn-LI-Small-H0-s1`, `nb3-tdgnn-LI-Small-H1-s1`, `nb3-tdgnn-LI-Small-H3600-s1`, `nb3-tdgnn-LI-Small-H86400-s1`, `nb3-tdgnn-LI-Small-Hinf-s1`, `nb3-tdgnn-HI-Small-H0-s1-attack`, `nb3-tdgnn-HI-Small-H1-s1-attack`, `nb3-tdgnn-HI-Small-H3600-s1-attack`

# NB3: TD-GNN across horizons

**Goal:** A learned model whose view can be restricted exactly, for fair per-horizon curves.

- **P0:** leakage check; 5 horizons and 3 seeds on HI-Small and LI-Small; timing attack at 3 horizons
- **P1:** remaining horizons; three ablations; attack on LI-Small; GPU service time

**Time:** 20 to 40 GPU hours (the least reliable estimate: no full run has finished).

## How to run

### On Kaggle
**Settings (right-hand panel):** Accelerator **GPU T4 x2**; Internet **on**.

**Attach as inputs:**
- `ealtman2019/ibm-transactions-for-anti-money-laundering-aml` (holds all four AMLworld datasets)
- The code package (a private dataset holding `aml-code-*.zip`)

### On Colab
**Runtime, Change runtime type:** **T4 GPU**. Keep this tab open: an idle or closed tab ends the session.

**One-time setup in Google Drive:** make a folder `aml` and put `aml-code-*.zip` in it. The notebook mounts Drive, downloads the raw files it needs, and keeps its output in `aml/out_nb3`. If a session disconnects, use **Runtime, Run all** again: finished jobs are not repeated, and interrupted ones continue from their checkpoints.

If the download is blocked, put the raw files in `aml/data` in Drive and the notebook copies them from there.

### Either platform
**Step 1, pilot (under 2 hours).** Leave `MODE = 'pilot'` and run all cells. Read `aml_pilot/PILOT_REPORT.md` at the end. Go on only if it says PASS.

**Step 2, production.** Set `MODE = 'production'` and run all cells. The run stops cleanly at the time budget.

**Step 3, continue.** If the last cell says jobs remain, run again. On Kaggle, first attach this notebook's output as an input (**Add Input, Notebook Output**).

**Step 4, hand over.** On Kaggle, publish the final output's `aml_run` folder as a private dataset named `aml-out-nb3-v1`. On Colab, share the `out_nb3` folder.

Keep `GPUS = 1`. On Kaggle, a pilot shows whether T4 x2 costs double; if it does not, `GPUS = 2` runs two jobs at once.

**Rules:** do not edit the code, change a job's settings or copy in a token. A job whose code or settings changed refuses to resume, and that is deliberate. Test metrics for sealed datasets are never shown here: only the merge step computes them.

In [ ]:
# Settings. These are the only values to change.
import os, sys
NOTEBOOK = 'NB3'
MODE = 'pilot'            # 'pilot' first; 'production' only after PILOT_REPORT.md says PASS
_COLAB = bool(os.environ.get('AMLH_PLATFORM') == 'colab' or 'google.colab' in sys.modules)
TIERS = ('P0',)   # P0 runs first. P0 is what the 15 November milestone needs
BUDGET_HOURS = 5.0 if _COLAB else 10.5   # stops cleanly. Colab sessions are shorter and less predictable than Kaggle's 12 h
GPUS = 1                  # number of GPU worker processes
SHARD = (0, 3)              # (i, n) runs every n-th job chain, for taking overflow on another machine
MIRROR_SLUG = None        # Kaggle only: 'account/private-dataset' for hourly copies; needs Kaggle Secrets KAGGLE_USERNAME and KAGGLE_KEY
RUN_TESTS = True          # a failing test stops the notebook before it spends quota
INSTALL = True
EXPECTED_CODE_HASH = '2849e1ebe66ecd10'

In [ ]:
# Where things live. Kaggle attaches datasets under /kaggle/input. Colab has none, so it downloads the raw files and keeps its output in Google Drive,
# which means a disconnected session resumes from where it stopped.
import os, sys
from pathlib import Path

PLATFORM = os.environ.get('AMLH_PLATFORM') or ('colab' if 'google.colab' in sys.modules else 'kaggle')
CONTENT = Path('/content')
DRIVE = CONTENT / 'drive' / 'MyDrive'
if PLATFORM == 'colab':
    if not DRIVE.is_dir():
        from google.colab import drive
        drive.mount(str(CONTENT / 'drive'))
    HOME = DRIVE / 'aml'                       # put the code package (aml-code-*.zip) here
    INPUT = CONTENT / 'aml_data'               # raw files, downloaded at the start of each session
    CODE_SEARCH = [HOME, CONTENT]
    OUTPUTS = HOME / 'collect'                 # the merge notebook reads every notebook's output from here
    WORKING = HOME / (f"out_{NOTEBOOK.lower()}" + (f"_shard{SHARD[0]}of{SHARD[1]}" if SHARD else ''))
    RESTORE = []                               # the output already stays in Drive
    WORKING.mkdir(parents=True, exist_ok=True)
else:
    INPUT = Path('/kaggle/input')
    CODE_SEARCH = [INPUT]
    OUTPUTS = INPUT
    WORKING = Path('/kaggle/working')
    RESTORE = None                             # attached earlier outputs are found under /kaggle/input
print(PLATFORM, 'input', INPUT, 'output', WORKING)

In [ ]:
# Unpacks the code package outside the output folder, so it is not exported. Kaggle may unzip an uploaded zip, so both forms are handled.
import glob, shutil, subprocess, sys, time, zipfile

CODE = Path('/tmp/aml_code')
shutil.rmtree(CODE, ignore_errors=True)
if PLATFORM == 'colab':
    found = {p.name: p for root in CODE_SEARCH for p in sorted(Path(root).glob('aml-code-*.zip'))}  # the same file in two places counts once
    zips, folders = [str(p) for p in found.values()], []
else:
    zips = glob.glob(str(INPUT / '**' / 'aml-code-*.zip'), recursive=True)
    folders = [p.parents[2] for p in INPUT.glob('**/src/amlh/runner.py')]
assert len(zips) + len(folders) == 1, f'attach or upload exactly one code package; found {zips + [str(f) for f in folders]}'
if zips:
    zipfile.ZipFile(zips[0]).extractall(CODE)
else:
    shutil.copytree(folders[0], CODE)
sys.path.insert(0, str(CODE / 'src'))

from amlh.runner import code_hash
assert code_hash(CODE) == EXPECTED_CODE_HASH, f'this notebook expects code {EXPECTED_CODE_HASH}, but the package is {code_hash(CODE)}'
if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-kaggle.txt')], check=True)
print('code version', EXPECTED_CODE_HASH)

In [ ]:
# Colab has no attached datasets, so this fetches the raw files (and checks each is whole). Kaggle already has them.
if PLATFORM == 'colab':
    from amlh.fetch import ensure_raw
    from amlh.session import datasets_of, pilot_specs
    if MODE == 'pilot':
        names = sorted({j.cfg['dataset'] for j in pilot_specs(NOTEBOOK)[0] if 'dataset' in j.cfg})
    else:
        names = datasets_of(NOTEBOOK, TIERS)
    print('raw files needed:', names)
    if MODE == 'pilot':   # a file that cannot be fetched is skipped and reported by the pilot, not fatal
        for n in names:
            try:
                ensure_raw([n], INPUT, backup=HOME / 'data')
            except Exception as exc:
                print('could not fetch', n, '-', exc)
    else:
        ensure_raw(names, INPUT, backup=HOME / 'data')

In [ ]:
from amlh.session import setup_environment, run_preflight_tests
print(setup_environment(NOTEBOOK) if INSTALL else 'install skipped')
if RUN_TESTS:
    run_preflight_tests(CODE)

In [ ]:
# Kaggle only. Tokens stay in Kaggle Secrets. They are never written to a file or printed.
if MIRROR_SLUG and PLATFORM == 'kaggle':
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')

In [ ]:
from amlh.session import print_summary, run_notebook, run_pilot, run_workers

start = time.time()
if MODE == 'pilot':
    report = run_pilot(NOTEBOOK, input_root=INPUT, working=WORKING)
    assert report['pass'], 'pilot failed: read aml_pilot/PILOT_REPORT.md before production'
elif GPUS > 1:
    run_workers(NOTEBOOK, GPUS, tiers=TIERS, budget_hours=BUDGET_HOURS, input_root=INPUT, working=WORKING, mirror_slug=MIRROR_SLUG, restore_roots=RESTORE)
else:
    run_notebook(NOTEBOOK, tiers=TIERS, budget_hours=BUDGET_HOURS, input_root=INPUT, working=WORKING, shard=SHARD, mirror_slug=MIRROR_SLUG, restore_roots=RESTORE)
print(f'session used {(time.time() - start) / 3600:.2f} h')
print_summary(WORKING)

## Final verification

Prints a pass or fail for each check, and what to do next.

In [ ]:
from amlh.session import final_checklist
if MODE == 'production':
    final_checklist(NOTEBOOK, TIERS, WORKING, SHARD, platform=PLATFORM)